# ODI to Databricks Migration: `insert_trg_dep.txt`

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook contains the converted Spark SQL logic from the ODI session `insert_trg_dep.txt`.
It performs an INSERT operation into the `workspace.hr.trg_dep` table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "ETL Last Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59", "ETL Current Extract Time (YYYY-MM-DD HH:MM:SS)")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Target Table DDL

Assumed DDL for `trg_dep` based on the INSERT statement. If this table already exists with a different schema, this statement will not alter it unless specified.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.hr.trg_dep (
  department_id   BIGINT,
  department_name STRING,
  manager_id      BIGINT,
  location_id     BIGINT
) USING DELTA;

## Source Table DDL (for context)

Assumed DDL for `departments` for demonstration purposes. This table is expected to exist.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.hr.departments (
  department_id   BIGINT,
  department_name STRING,
  manager_id      BIGINT,
  location_id     BIGINT
) USING DELTA;

## Insert Data into Target (SCEN_TASK_NO in {10} to {30} - consolidated)

The original `SCEN_TASK_NO` blocks {10}, {20}, {30} were empty and are consolidated here. This task inserts all records from the source `departments` table into the target `trg_dep` table.

**Original ODI Hint Removal:** `/*+ APPEND PARALLEL */` has been removed as it's an Oracle-specific hint not applicable to Databricks Delta Lake.

In [ ]:
%sql
INSERT INTO workspace.hr.trg_dep (
  department_id,
  department_name,
  manager_id,
  location_id
)
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS trg_dep_record_count
FROM workspace.hr.trg_dep;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** All schema references (`HR`) have been converted to `workspace.hr` (lowercase, stripped environment suffixes). Table names (`TRG_DEP`, `DEPARTMENTS`) have been converted to lowercase (`trg_dep`, `departments`).
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable in Databricks Spark SQL / Delta Lake.
3.  **DDL Assumption:** The `CREATE TABLE IF NOT EXISTS` statements for `workspace.hr.trg_dep` and `workspace.hr.departments` are based on an assumed schema from typical Oracle HR `DEPARTMENTS` table structure. Please verify these DDLs against your actual source system metadata and adjust column types if necessary. Specifically:
    *   `DEPARTMENT_ID`: Assumed `BIGINT`.
    *   `DEPARTMENT_NAME`: Assumed `STRING`.
    *   `MANAGER_ID`: Assumed `BIGINT`.
    *   `LOCATION_ID`: Assumed `BIGINT`.
4.  **ETL Parameters:** Placeholder widgets have been created for common ODI global parameters. While not directly used in this specific simple INSERT, they are included for consistency with standard ETL framework requirements.
5.  **Empty SCEN_TASK_NOs:** The original `SCEN_TASK_NO in {10}`, `{20}`, `{30}` blocks were empty and are not represented as separate cells. The `INSERT` statement follows them directly.